In [1]:
from src.models.embedding import EmbeddingModel
from src.config import Constants

embedding_model = EmbeddingModel(Constants.EMBEDDING_MODEL)

/home/cnguyen/do_an/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model from local path: ./models/embedding/ai-forever/FRIDA


In [2]:
# !pip install -q rouge nltk

In [24]:
from typing import List, Dict
from rouge import Rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sentence_transformers import util
import json

file_path = "./data/results.json"
# Load JSON data
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

In [25]:
len(data), data[:5]

(428,
 [{'question': 'Какие предметы требуются для поступления на англоязычную программу «Computer Science for foreign citizens»?',
   'answer': 'Для поступления на англоязычную программу «Computer Science for foreign citizens» требуются следующие предметы: математика и информатика.',
   'ground_true_answer': 'Математика и информатика. Русский язык не требуется.\nПрограмма предназначена только для иностранных граждан и только на платной основе.',
   'ground_true_source': 'bachelor_and_master_2025_rules',
   'retrieved_documents': [{'text': 'Программы вступительных испытаний\n\nКонкурсные группы с указанием перечня вступительных испытаний и их приоритета: Физтех-школа прикладной математики и информатики (ФПМИ)\n\nКонкурсная группа | Вступительные испытания (по приоритету)\nНаправление 01.03.02 Прикладная математика и информатика\nПрограммы, реализуемые на русском языке\nПрикладная математика и информатика | Математика, информатика, русский язык\nПроектирование и разработка комплексных б

In [38]:
def drop_empty_answers(data: List[Dict]) -> List[Dict]:
    return [a for a in data if a.get("answer") and a.get("ground_true_answer") and a.get("question")]

In [39]:
cleaned_data = drop_empty_answers(data)
len(cleaned_data)

427

**BLEU and ROUGH**

In [33]:
# Initialize metrics
rouge = Rouge()
bleu_scores = []
rouge_scores = []

# Calculate metrics
for entry in cleaned_data:
    reference = entry["ground_true_answer"]
    hypothesis = entry["answer"]

    # Skip if reference is empty
    if not reference:
        continue

    # BLEU score
    smoothie = SmoothingFunction().method4
    bleu = sentence_bleu([reference.split()], hypothesis.split(), smoothing_function=smoothie)
    bleu_scores.append(bleu)

    # ROUGE score
    rouge_score = rouge.get_scores(hypothesis, reference)[0]
    rouge_scores.append(rouge_score)

# Average scores
average_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
average_rouge = {
    "rouge-1": {
        "f": sum(score["rouge-1"]["f"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "p": sum(score["rouge-1"]["p"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "r": sum(score["rouge-1"]["r"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
    },
    "rouge-2": {
        "f": sum(score["rouge-2"]["f"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "p": sum(score["rouge-2"]["p"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "r": sum(score["rouge-2"]["r"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
    },
    "rouge-l": {
        "f": sum(score["rouge-l"]["f"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "p": sum(score["rouge-l"]["p"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
        "r": sum(score["rouge-l"]["r"] for score in rouge_scores) / len(rouge_scores) if rouge_scores else 0,
    },
}

average_bleu, average_rouge

(0.047828310921452996,
 {'rouge-1': {'f': 0.21697527665867533,
   'p': 0.16321211661267634,
   'r': 0.41941298526128123},
  'rouge-2': {'f': 0.09817091249008952,
   'p': 0.07264288349315295,
   'r': 0.2087163105321249},
  'rouge-l': {'f': 0.20534235298537323,
   'p': 0.15424827694595114,
   'r': 0.39961971381684236}})

**AVG SIMILARITY**

In [ ]:
# Подсчёт семантического сходства между ответами модели и правильными ответами
similarities = []
for entry in cleaned_data:
    reference = entry["ground_true_answer"]
    hypothesis = entry["answer"]
    emb_ref = embedding_model.embed(reference, convert_to_tensor=True)
    emb_hyp = embedding_model.embed(hypothesis, convert_to_tensor=True)
    similarity = float(util.cos_sim(emb_ref, emb_hyp))
    similarities.append(similarity)

# Среднее значение семантического сходства
average_similarity = sum(similarities) / len(similarities)

print(f"Average similary: {average_similarity:.3f}")

0.5471434679746069

**Context precision**

In [35]:
relevant = 0
total = 0

for entry in cleaned_data:
    gt = entry["ground_true_answer"]
    gt_emb = embedding_model.embed(reference, convert_to_tensor=True)

    for doc in entry["retrieved_documents"]:
        doc_emb = embedding_model.embed(reference, convert_to_tensor=True)
        sim = util.cos_sim(gt_emb, doc_emb).item()
        if sim > 0.7:
            relevant += 1
        total += 1

print(f"Context Precision (SNR, semantically): {relevant / total:.3f}")

Context Precision (SNR, semantically): 1.000


**Context recall**

In [50]:
total = len(data)
relevant_found = 0

for entry in cleaned_data:
    gt = entry["ground_true_answer"]
    gt_emb = embedding_model.embed(gt, convert_to_tensor=True)
    found = False
    for doc in entry["retrieved_documents"]:
        doc_emb = embedding_model.embed(doc["text"], convert_to_tensor=True)
        sim = util.cos_sim(gt_emb, doc_emb).item()
        if sim > 0.45:
            found = True
            break
    if found:
        relevant_found += 1

print(f"Context Recall (semantic): {relevant_found / total:.3f}")

Context Recall (semantic): 0.561


**Faithfulness**

In [48]:
faithful = 0
total = 0

for entry in cleaned_data:
    ans = entry["answer"]
    ans_emb = embedding_model.embed(ans, convert_to_tensor=True)

    supported = False
    for doc in entry["retrieved_documents"]:
        doc_emb = embedding_model.embed(doc["text"], convert_to_tensor=True)
        sim = util.cos_sim(ans_emb, doc_emb).item()
        if sim > 0.7:
            supported = True
            break

    if supported:
        faithful += 1
    total += 1

print(f"Faithfulness (semantic): {faithful / total:.3f}")

Faithfulness (semantic): 0.541


**Answer Relevance**

In [40]:
relevance_scores = []

for entry in cleaned_data:
    question = entry["question"]
    answer = entry["answer"]

    emb_q = embedding_model.embed(question, convert_to_tensor=True)
    emb_a = embedding_model.embed(answer, convert_to_tensor=True)

    similarity = util.cos_sim(emb_q, emb_a).item()
    relevance_scores.append(similarity)

average_relevance = sum(relevance_scores) / len(relevance_scores)
print(f"Average Answer Relevance: {average_relevance:.3f}")

Average Answer Relevance: 0.688


**Answer similarity**

In [41]:
similarities = []

for entry in cleaned_data:
    answer = entry["answer"]
    ground_truth = entry["ground_true_answer"]

    emb1 = embedding_model.embed(answer, convert_to_tensor=True)
    emb2 = embedding_model.embed(ground_truth, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()
    similarities.append(similarity)

average_similarity = sum(similarities) / len(similarities)
print(f"Answer Similarity (semantic): {average_similarity:.3f}")

Answer Similarity (semantic): 0.547


**Answer correctness**

In [47]:
correct_answers = 0

for entry in cleaned_data:
    answer = entry["answer"]
    ground_truth = entry["ground_true_answer"]

    emb1 = embedding_model.embed(answer, convert_to_tensor=True)
    emb2 = embedding_model.embed(ground_truth, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()
    if similarity > 0.5:
        correct_answers += 1

correctness_score = correct_answers / len(data)
print(f"Answer Correctness (semantic threshold): {correctness_score:.3f}")

Answer Correctness (semantic threshold): 0.596
